# Week 8: NumPy, pandas, and Data Exploration
## IT 2012 - Unstructured Data


## Learning objectives

By the end of this notebook you will be able to:

1. Create NumPy arrays and apply vectorized operations instead of Python loops
2. Build pandas `Series` and `DataFrame` objects from Python structures
3. **Load data that looks like a MongoDB cursor** — a `list[dict]` or a JSON file — into a `DataFrame`
4. Read large CSV files in chunks when they do not fit in memory
5. Inspect and reduce memory usage by optimising `dtypes`
6. Explore a new dataset with `head`, `describe`, `info`, `value_counts`
7. Select data with `loc` / `iloc` and filter with boolean conditions
8. Detect and count missing values
9. Use regular expressions on text columns via the `.str` accessor

---
**Pipeline context.** In Weeks 4–7 we extracted content from unstructured sources (PDFs, images via OCR, audio transcripts, web pages). In a real pipeline those extractions are written to MongoDB. This week we take that extracted content and start exploring it with pandas.

Because we are not connecting to MongoDB in this demo, we simulate the cursor output with a JSON file (`extracted_documents.json`) — the shape is identical to what `collection.find()` would return.

## 0 — Setup

In [1]:
# Core scientific stack
import numpy as np
import pandas as pd
import json
import re
from pathlib import Path

# Display settings — makes exploration more readable
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("numpy :", np.__version__)
print("pandas:", pd.__version__)

numpy : 2.3.5
pandas: 2.3.3


---
## 1 — NumPy basics

NumPy is the foundation of scientific Python. pandas is built on top of it. Every pandas `Series` or column is, internally, a NumPy array.

**Why NumPy?** Two reasons: (1) *vectorised* operations — you write `a + b` instead of a `for` loop — and (2) *performance* — those operations run in C, not Python.

### 1.1 Creating arrays

In [2]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("a         =", a)
print("type      =", type(a))

# Filled arrays
print("zeros(3)  =", np.zeros(3))
print("ones(3)   =", np.ones(3))

# Ranges
print("arange    =", np.arange(0, 10, 2))           # start, stop, step
print("linspace  =", np.linspace(0, 1, 5))          # 5 evenly spaced points

a         = [1 2 3 4 5]
type      = <class 'numpy.ndarray'>
zeros(3)  = [0. 0. 0.]
ones(3)   = [1. 1. 1.]
arange    = [0 2 4 6 8]
linspace  = [0.   0.25 0.5  0.75 1.  ]


### 1.2 Array attributes

In [3]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])

print("shape :", matrix.shape)   # (rows, cols)
print("ndim  :", matrix.ndim)    # number of dimensions
print("size  :", matrix.size)    # total number of elements
print("dtype :", matrix.dtype)   # data type of elements

shape : (2, 3)
ndim  : 2
size  : 6
dtype : int64


### 1.3 Indexing and slicing

NumPy indexing is like Python lists, but extends to multiple dimensions.

In [4]:
arr = np.array([10, 20, 30, 40, 50])

print("arr[0]    :", arr[0])       # first element
print("arr[-1]   :", arr[-1])      # last element
print("arr[1:4]  :", arr[1:4])     # slice

# 2D: [row, col]
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("m[0, 1]   :", m[0, 1])      # row 0, col 1
print("m[:, 0]   :", m[:, 0])      # whole first column
print("m[1, :]   :", m[1, :])      # whole second row

arr[0]    : 10
arr[-1]   : 50
arr[1:4]  : [20 30 40]
m[0, 1]   : 2
m[:, 0]   : [1 4 7]
m[1, :]   : [4 5 6]


### 1.4 Vectorised operations — the big idea

Instead of writing a `for` loop, apply the operation to the *whole array at once*. The result is shorter code and much faster execution.

In [5]:
prices_kam = np.array([245.60, 15.40, 12.00, 3200.00, 87.50])

# Convert KM (Bosnian Mark) to EUR — one line, no loop
KM_TO_EUR = 0.511
prices_eur = prices_kam * KM_TO_EUR
print("KM  :", prices_kam)
print("EUR :", np.round(prices_eur, 2))

# Element-wise math between arrays of the same shape
tax = np.array([0.17, 0.17, 0.17, 0.17, 0.17])
with_vat = prices_kam * (1 + tax)
print("With VAT:", np.round(with_vat, 2))

KM  : [ 245.6   15.4   12.  3200.    87.5]
EUR : [ 125.5     7.87    6.13 1635.2    44.71]
With VAT: [ 287.35   18.02   14.04 3744.    102.38]


### 1.5 Broadcasting

When shapes do not match exactly, NumPy *broadcasts* — stretches the smaller array so the operation makes sense. A scalar broadcasts against any array. A 1-D array can broadcast across rows or columns of a 2-D array.

In [6]:
# Scalar broadcast: 0.17 is stretched to the shape of prices_kam
with_vat_v2 = prices_kam * 1.17
print(np.round(with_vat_v2, 2))

# 1-D broadcast across rows of a 2-D matrix
grades = np.array([[85, 90, 78],
                   [60, 72, 88],
                   [95, 91, 84]])
bonus = np.array([2, 3, 1])       # one bonus per column (assignment)

print("Grades + bonus per column:")
print(grades + bonus)

[ 287.35   18.02   14.04 3744.    102.38]
Grades + bonus per column:
[[87 93 79]
 [62 75 89]
 [97 94 85]]


### 1.6 Speed: vectorised vs Python loop

This is the punch-line — why we bother with NumPy at all.

In [7]:
import time

N = 1_000_000
py_list  = list(range(N))
np_array = np.arange(N)

# Python loop
t0 = time.perf_counter()
result_py = [x * 2 for x in py_list]
t_py = time.perf_counter() - t0

# Vectorised NumPy
t0 = time.perf_counter()
result_np = np_array * 2
t_np = time.perf_counter() - t0

print(f"Python list comprehension : {t_py*1000:7.2f} ms")
print(f"NumPy vectorised          : {t_np*1000:7.2f} ms")
print(f"Speed-up                  : {t_py/t_np:7.1f}x")

Python list comprehension :   17.60 ms
NumPy vectorised          :    1.35 ms
Speed-up                  :    13.0x


---
## 2 — pandas `Series`

A `Series` is a 1-D labelled array. Think of it as a NumPy array with an **index**.

In [8]:
# From a Python list — default integer index
s = pd.Series([10, 20, 30, 40])
print(s)
print("dtype :", s.dtype)

0    10
1    20
2    30
3    40
dtype: int64
dtype : int64


In [9]:
# From a dictionary — keys become the index
languages = pd.Series({"en": 24, "bs": 18, "hr": 9, "sr": 6, "de": 3})
languages.name = "document_count"
languages.index.name = "language"

print(languages)
print()
print("Label-based access :", languages["bs"])
print("Position-based     :", languages.iloc[0])

language
en    24
bs    18
hr     9
sr     6
de     3
Name: document_count, dtype: int64

Label-based access : 18
Position-based     : 24


In [10]:
# Series operations are vectorised, just like NumPy
print("Total    :", languages.sum())
print("Mean     :", round(languages.mean(), 2))
print("Max      :", languages.max())
print("Sorted   :")
print(languages.sort_values(ascending=False))

Total    : 60
Mean     : 12.0
Max      : 24
Sorted   :
language
en    24
bs    18
hr     9
sr     6
de     3
Name: document_count, dtype: int64


---
## 3 — pandas `DataFrame`

A `DataFrame` is a 2-D labelled table. Rows have an index, columns have names. Each column is a `Series`.

### 3.1 From a dict of lists

In [11]:
df_small = pd.DataFrame({
    "filename":    ["receipt_001.pdf", "notes.png",  "call.mp3"],
    "source_type": ["pdf",             "image",      "audio"],
    "word_count":  [12,                7,            245],
})
df_small

,filename,source_type,word_count
0,receipt_001.pdf,pdf,12
1,notes.png,image,7
2,call.mp3,audio,245


### 3.2 From a list of dicts — this is what a MongoDB cursor returns

When you call `list(collection.find())` in PyMongo, you get exactly this — a list of dictionaries. pandas can read it directly.

In [12]:
# Exactly the shape you'd get from: list(db.documents.find())
cursor_like = [
    {"_id": "a1", "filename": "receipt_001.pdf", "source_type": "pdf",   "word_count": 12},
    {"_id": "a2", "filename": "notes.png",       "source_type": "image", "word_count": 7},
    {"_id": "a3", "filename": "call.mp3",        "source_type": "audio", "word_count": 245},
]

df_from_cursor = pd.DataFrame(cursor_like)
df_from_cursor

,_id,filename,source_type,word_count
0,a1,receipt_001.pdf,pdf,12
1,a2,notes.png,image,7
2,a3,call.mp3,audio,245


### 3.3 DataFrame attributes

In [13]:
print("shape   :", df_from_cursor.shape)
print("ndim    :", df_from_cursor.ndim)
print("size    :", df_from_cursor.size)
print("columns :", list(df_from_cursor.columns))
print("index   :", list(df_from_cursor.index))
print()
print("dtypes:")
print(df_from_cursor.dtypes)

shape   : (3, 4)
ndim    : 2
size    : 12
columns : ['_id', 'filename', 'source_type', 'word_count']
index   : [0, 1, 2]

dtypes:
_id            object
filename       object
source_type    object
word_count      int64
dtype: object


---
## 4 — Loading data

This section is the core of the demo. We load data **as if it came from MongoDB**, but read from a JSON file instead.

### 4.1 Load JSON that represents a MongoDB collection dump

In [14]:
# In a real project, this block would be:
#
#     from pymongo import MongoClient
#     client = MongoClient("mongodb://localhost:27017")
#     docs = list(client.unstructured.documents.find())
#
# For this demo we read an equivalent JSON export instead.

with open("extracted_documents.json", encoding="utf-8") as f:
    docs = json.load(f)

print("Type of docs        :", type(docs).__name__)
print("Number of documents :", len(docs))
print("First document keys :", list(docs[0].keys()))

Type of docs        : list
Number of documents : 60
First document keys : ['_id', 'source_type', 'filename', 'title', 'extracted_at', 'file_size_kb', 'language', 'status', 'content', 'metadata', 'tags']


### 4.2 Two ways to load into a DataFrame

**Option A — `pd.DataFrame(list_of_dicts)`** — keeps nested fields as dict/list cells. Fast, but nested fields are awkward to query.

**Option B — `pd.json_normalize`** — flattens nested fields into `content.text`, `metadata.pages`, ... columns. Usually what you want for analysis.

In [15]:
# Option A — nested columns stay as Python objects
df_nested = pd.DataFrame(docs)
print("Option A columns:", list(df_nested.columns))
df_nested.head(3)

Option A columns: ['_id', 'source_type', 'filename', 'title', 'extracted_at', 'file_size_kb', 'language', 'status', 'content', 'metadata', 'tags']


,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,content,metadata,tags
0,507f1f77bcf86cd799439000,pdf,sarajevo_tourism_report_2024.pdf,Sarajevo Tourism Report 2024,2025-09-04T10:13:00,1119.9,de,processed,{'text': 'Abstract: This paper explores NLP te...,"{'pages': 3, 'duration_sec': None, 'ocr_confid...","[report, tourism]"
1,507f1f77bcf86cd799439001,image,product_label.png,product_label,2025-09-01T13:44:00,2100.5,hr,partial,"{'text': 'Best Before 12/2025 Made in BiH', 'w...","{'pages': None, 'duration_sec': None, 'ocr_con...","[id, personal]"
2,507f1f77bcf86cd799439002,audio,interview_student.mp3,interview_student,2025-10-16T03:16:00,1686.3,en,processed,{'text': 'So my thesis is about multimodal dec...,"{'pages': None, 'duration_sec': 1311.6, 'ocr_c...",[meeting]


In [16]:
# Option B — flattened with json_normalize (the pandas way)
df = pd.json_normalize(docs)
print("Option B columns:", list(df.columns))
df.head(3)

Option B columns: ['_id', 'source_type', 'filename', 'title', 'extracted_at', 'file_size_kb', 'language', 'status', 'tags', 'content.text', 'content.word_count', 'metadata.pages', 'metadata.duration_sec', 'metadata.ocr_confidence', 'metadata.author']


,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,tags,content.text,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence,metadata.author
0,507f1f77bcf86cd799439000,pdf,sarajevo_tourism_report_2024.pdf,Sarajevo Tourism Report 2024,2025-09-04T10:13:00,1119.9,de,processed,"[report, tourism]",Abstract: This paper explores NLP techniques f...,9,3.0,NaN,NaN,Alex Johnson
1,507f1f77bcf86cd799439001,image,product_label.png,product_label,2025-09-01T13:44:00,2100.5,hr,partial,"[id, personal]",Best Before 12/2025 Made in BiH,6,NaN,NaN,0.906,None
2,507f1f77bcf86cd799439002,audio,interview_student.mp3,interview_student,2025-10-16T03:16:00,1686.3,en,processed,[meeting],So my thesis is about multimodal deception det...,9,NaN,1311.6,NaN,None


Notice what `json_normalize` did:

- `content.text` and `content.word_count` are now top-level columns
- `metadata.pages`, `metadata.duration_sec`, `metadata.ocr_confidence`, `metadata.author` likewise
- `tags` is still a list because it is an array, not an object

This is the shape we will work with for the rest of the notebook.

### 4.3 Parse the datetime column

Right now `extracted_at` is a plain string. Convert it to a proper datetime so we can do time-based operations.

In [17]:
df["extracted_at"] = pd.to_datetime(df["extracted_at"])
print(df["extracted_at"].dtype)
df["extracted_at"].head()

datetime64[ns]


0   2025-09-04 10:13:00
1   2025-09-01 13:44:00
2   2025-10-16 03:16:00
3   2025-09-07 05:14:00
4   2025-11-25 16:44:00
Name: extracted_at, dtype: datetime64[ns]

### 4.4 Other common readers (same pattern)

```python
pd.read_csv("file.csv")                   # sep, header, names, dtype, parse_dates, na_values, encoding
pd.read_json("file.json", orient="records")
pd.read_excel("file.xlsx", sheet_name="Sheet1")
```

In [18]:
# Quick demo of read_csv with a few key parameters
sample_csv = '''name;age;joined
Emina;34;2019-09-01
Marko;28;2021-03-15
Amra;41;2015-06-10'''

from io import StringIO

df_csv = pd.read_csv(
    StringIO(sample_csv),
    sep=";",                    # custom separator
    parse_dates=["joined"],     # parse as datetime
)
df_csv

,name,age,joined
0,Emina,34,2019-09-01
1,Marko,28,2021-03-15
2,Amra,41,2015-06-10


---
## 5 — Loading large files in chunks

When a CSV is too big to fit in memory, read it a piece at a time with `chunksize`. Each chunk is itself a `DataFrame`.

In [19]:
# Build a medium-sized CSV so we have something to chunk.
# In real life this file would be on disk already, produced by an earlier
# pipeline step (e.g. dumped from MongoDB or downloaded from S3).
np.random.seed(0)
N = 50_000
big_df = pd.DataFrame({
    "doc_id":      [f"doc_{i:06d}" for i in range(N)],
    "source_type": np.random.choice(["pdf", "image", "audio", "web"], N),
    "word_count":  np.random.randint(0, 5000, N),
    "file_size_kb": np.round(np.random.uniform(5, 10000, N), 1),
})
big_df.to_csv("big_documents.csv", index=False)
print("File written. Rows:", len(big_df))

File written. Rows: 50000


In [20]:
# Process the file 10 000 rows at a time and aggregate as we go
running_total_words = 0
chunks_seen = 0
per_type_words = {}

reader = pd.read_csv("big_documents.csv", chunksize=10_000)

for chunk in reader:
    chunks_seen += 1
    running_total_words += int(chunk["word_count"].sum())

    # Aggregate by source_type within this chunk, then merge
    for src, n in chunk.groupby("source_type")["word_count"].sum().items():
        per_type_words[src] = per_type_words.get(src, 0) + int(n)

print(f"Chunks processed  : {chunks_seen}")
print(f"Total words       : {running_total_words:,}")
print(f"Words by source   : {per_type_words}")

Chunks processed  : 5
Total words       : 125,203,521
Words by source   : {'audio': 31000517, 'image': 31667551, 'pdf': 31442292, 'web': 31093161}


**When to use chunking**
- The file is bigger than RAM (or close to it)
- You only need an aggregate (sum, mean, count), not the raw rows
- You are filtering heavily and keeping only a small fraction

**When NOT to use chunking**
- The file fits in memory — just use `pd.read_csv`
- You need cross-row operations (sorting the whole table, joins)

---
## 6 — Memory management

`DataFrame.info()` and `DataFrame.memory_usage()` tell you how much memory your table uses, column by column. Two cheap wins: **downcast numeric columns** and **convert repeated strings to `category`**.

In [21]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   _id                      60 non-null     object        
 1   source_type              60 non-null     object        
 2   filename                 60 non-null     object        
 3   title                    60 non-null     object        
 4   extracted_at             60 non-null     datetime64[ns]
 5   file_size_kb             58 non-null     float64       
 6   language                 58 non-null     object        
 7   status                   60 non-null     object        
 8   tags                     60 non-null     object        
 9   content.text             60 non-null     object        
 10  content.word_count       60 non-null     int64         
 11  metadata.pages           14 non-null     float64       
 12  metadata.duration_sec    15 non-null  

In [22]:
# Column-by-column memory, in KB
mem_kb = df.memory_usage(deep=True) / 1024
mem_kb.round(2)

Index                      0.13
_id                        4.28
source_type                3.10
filename                   4.36
title                      4.09
extracted_at               0.47
file_size_kb               0.47
language                   2.94
status                     3.35
tags                       5.50
content.text               6.92
content.word_count         0.47
metadata.pages             0.47
metadata.duration_sec      0.47
metadata.ocr_confidence    0.47
metadata.author            2.03
dtype: float64

### 6.1 Downcast numeric columns

In [23]:
# Before: default int64 / float64
print("Before downcast:")
print(df[["content.word_count", "file_size_kb"]].dtypes)
before = df.memory_usage(deep=True).sum()

df["content.word_count"] = pd.to_numeric(df["content.word_count"], downcast="integer")
df["file_size_kb"]       = pd.to_numeric(df["file_size_kb"],       downcast="float")

print("\nAfter downcast:")
print(df[["content.word_count", "file_size_kb"]].dtypes)
after = df.memory_usage(deep=True).sum()
print(f"\nMemory: {before/1024:.2f} KB -> {after/1024:.2f} KB  (saved {(before-after)/1024:.2f} KB)")

Before downcast:
content.word_count      int64
file_size_kb          float64
dtype: object

After downcast:
content.word_count       int8
file_size_kb          float32
dtype: object

Memory: 39.50 KB -> 38.85 KB  (saved 0.64 KB)


### 6.2 Convert repeated strings to `category`

`source_type`, `language`, and `status` each have only a handful of unique values repeated many times. `category` stores them as integer codes — much smaller and faster to filter.

In [24]:
before = df.memory_usage(deep=True).sum()

for col in ["source_type", "language", "status"]:
    df[col] = df[col].astype("category")

after = df.memory_usage(deep=True).sum()
print("dtypes now:")
print(df.dtypes)
print(f"\nMemory: {before/1024:.2f} KB -> {after/1024:.2f} KB")

dtypes now:
_id                                object
source_type                      category
filename                           object
title                              object
extracted_at               datetime64[ns]
file_size_kb                      float32
language                         category
status                           category
tags                               object
content.text                       object
content.word_count                   int8
metadata.pages                    float64
metadata.duration_sec             float64
metadata.ocr_confidence           float64
metadata.author                    object
dtype: object

Memory: 38.85 KB -> 30.71 KB


---
## 7 — Basic exploration

When you meet a new dataset, always run the same checklist: shape, dtypes, head, describe, info, value_counts.

### 7.1 Shape and structure

In [25]:
print("shape   :", df.shape)
print("size    :", df.size)
print("ndim    :", df.ndim)
print("columns :", list(df.columns))

shape   : (60, 15)
size    : 900
ndim    : 2
columns : ['_id', 'source_type', 'filename', 'title', 'extracted_at', 'file_size_kb', 'language', 'status', 'tags', 'content.text', 'content.word_count', 'metadata.pages', 'metadata.duration_sec', 'metadata.ocr_confidence', 'metadata.author']


### 7.2 First / last / random rows

In [26]:
df.head(3)

,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,tags,content.text,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence,metadata.author
0,507f1f77bcf86cd799439000,pdf,sarajevo_tourism_report_2024.pdf,Sarajevo Tourism Report 2024,2025-09-04 10:13:00,1119.900024,de,processed,"[report, tourism]",Abstract: This paper explores NLP techniques f...,9,3.0,NaN,NaN,Alex Johnson
1,507f1f77bcf86cd799439001,image,product_label.png,product_label,2025-09-01 13:44:00,2100.500000,hr,partial,"[id, personal]",Best Before 12/2025 Made in BiH,6,NaN,NaN,0.906,None
2,507f1f77bcf86cd799439002,audio,interview_student.mp3,interview_student,2025-10-16 03:16:00,1686.300049,en,processed,[meeting],So my thesis is about multimodal deception det...,9,NaN,1311.6,NaN,None


In [27]:
df.tail(3)

,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,tags,content.text,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence,metadata.author
57,507f1f77bcf86cd799439057,web,klix.ba_article_-_politika.html,klix.ba article - politika,2025-10-03 23:07:00,2349.600098,bs,processed,[reference],Sarajevo — Nova mjera Vijeća ministara usvojen...,9,NaN,NaN,NaN,None
58,507f1f77bcf86cd799439058,web,klix.ba_article_-_politika.html,klix.ba article - politika,2025-09-11 15:07:00,272.799988,de,failed,[reference],Regional stability remains a key concern for E...,9,NaN,NaN,NaN,None
59,507f1f77bcf86cd799439059,image,handwritten_notes.png,handwritten_notes,2025-11-19 09:39:00,2216.399902,sr,processed,"[notes, meeting]",Ćevapi 10KM Burek 8KM Kafa 2KM,6,NaN,NaN,0.684,None


In [28]:
df.sample(3, random_state=1)

,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,tags,content.text,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence,metadata.author
39,507f1f77bcf86cd799439039,audio,voice_memo_meeting.mp3,voice_memo_meeting,2025-09-22 07:55:00,4460.600098,de,partial,[meeting],"Hello, I'd like to report an issue with my rec...",11,NaN,1621.9,NaN,None
41,507f1f77bcf86cd799439041,audio,customer_call_support.mp3,customer_call_support,2025-10-23 18:34:00,1160.199951,bs,partial,[meeting],"Dobar dan svima, danas ćemo govoriti o neurons...",9,NaN,259.3,NaN,None
2,507f1f77bcf86cd799439002,audio,interview_student.mp3,interview_student,2025-10-16 03:16:00,1686.300049,en,processed,[meeting],So my thesis is about multimodal deception det...,9,NaN,1311.6,NaN,None


### 7.3 Statistical summary

In [29]:
# describe() — numeric columns by default
df.describe()

,extracted_at,file_size_kb,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence
count,60,58.000000,60.000000,14.000000,15.000000,12.000000
mean,2025-10-31 07:14:39,2311.224121,8.433333,13.642857,1600.480000,0.752583
min,2025-09-01 13:44:00,13.400000,6.000000,1.000000,57.900000,0.644000
25%,2025-09-28 19:47:30,1016.724976,7.000000,6.750000,772.900000,0.675000
50%,2025-10-28 22:28:00,2158.449951,9.000000,15.500000,1608.500000,0.716000
75%,2025-12-03 06:13:15,3510.524902,9.000000,19.750000,2509.900000,0.830250
max,2025-12-31 07:47:00,4818.899902,11.000000,25.000000,3474.100000,0.906000
std,NaN,1481.854004,1.477133,7.581426,1011.512195,0.095505


In [30]:
# describe on object / category columns
df.describe(include=["object", "category"])

,_id,source_type,filename,title,language,status,tags,content.text,metadata.author
count,60,60,60,60,58,60,60,60,17
unique,60,4,24,24,5,3,20,20,8
top,507f1f77bcf86cd799439000,web,avaz.ba_news.html,avaz.ba news,hr,processed,[meeting],Sarajevo — Nova mjera Vijeća ministara usvojen...,Amra S.
freq,1,19,5,5,14,40,5,7,4


### 7.4 Frequency distributions

In [31]:
# How many docs per source type?
df["source_type"].value_counts()

source_type
web      19
audio    15
pdf      14
image    12
Name: count, dtype: int64

In [32]:
# Unique values / counts
print("Unique languages :", df["language"].nunique())
print("Unique statuses  :", df["status"].nunique())
print()
print("Status distribution (with NaN shown):")
print(df["status"].value_counts(dropna=False))

Unique languages : 5
Unique statuses  : 3

Status distribution (with NaN shown):
status
processed    40
partial      12
failed        8
Name: count, dtype: int64


---
## 8 — Selecting data

Three common tools: `df["col"]` for columns, `df.loc[...]` for **label-based** selection, `df.iloc[...]` for **integer-position** selection. `loc` and `iloc` are *not* interchangeable.

### 8.1 Column selection

In [33]:
# Single column -> Series
df["title"].head(3)

0    Sarajevo Tourism Report 2024
1                   product_label
2               interview_student
Name: title, dtype: object

In [34]:
# Multiple columns -> DataFrame (note the double brackets)
df[["title", "source_type", "content.word_count"]].head(3)

,title,source_type,content.word_count
0,Sarajevo Tourism Report 2024,pdf,9
1,product_label,image,6
2,interview_student,audio,9


### 8.2 Row selection with `loc` (label-based)

In [35]:
# By index label — our index is 0..59 so labels and positions look the same,
# but loc uses LABELS. Slice endpoint is inclusive with loc.
df.loc[0:2, ["title", "source_type"]]

,title,source_type
0,Sarajevo Tourism Report 2024,pdf
1,product_label,image
2,interview_student,audio


### 8.3 Row selection with `iloc` (position-based)

In [36]:
# iloc uses POSITIONS (0-based). Slice endpoint is exclusive — like Python lists.
df.iloc[0:3, [1, 2, 3]]

,source_type,filename,title
0,pdf,sarajevo_tourism_report_2024.pdf,Sarajevo Tourism Report 2024
1,image,product_label.png,product_label
2,audio,interview_student.mp3,interview_student


**Rule of thumb**
- Use `loc` when you have meaningful row labels (dates, IDs) or are filtering with a boolean mask
- Use `iloc` when you want "the first 5 rows" regardless of what the labels are

---
## 9 — Filtering

### 9.1 Single condition

In [37]:
# Only PDF documents
pdf_docs = df[df["source_type"] == "pdf"]
print(f"{len(pdf_docs)} PDF documents")
pdf_docs[["title", "metadata.pages", "content.word_count"]].head()

14 PDF documents


,title,metadata.pages,content.word_count
0,Sarajevo Tourism Report 2024,3.0,9
3,Contract Agreement v3,19.0,7
8,Quarterly Sales Summary,13.0,10
10,Annual Budget Proposal,18.0,9
12,Invoice 00123,20.0,10


### 9.2 Multiple conditions — `&`, `|`, `~`

Use `&` for AND, `|` for OR, `~` for NOT. **Every condition must be wrapped in parentheses** because these operators bind tighter than comparison operators.

In [38]:
# English PDFs with more than 5 words
mask = (df["source_type"] == "pdf") & (df["language"] == "en") & (df["content.word_count"] > 5)
df.loc[mask, ["title", "language", "content.word_count"]]

,title,language,content.word_count
8,Quarterly Sales Summary,en,10
31,Quarterly Sales Summary,en,9
42,Meeting Minutes - IT Dept,en,7


In [39]:
# Everything that is NOT processed successfully
not_processed = df[~(df["status"] == "processed")]
not_processed[["title", "source_type", "status"]].head()

,title,source_type,status
1,product_label,image,partial
7,whiteboard_photo,image,partial
14,receipt_scan_001,image,failed
16,hackernews discussion,web,failed
17,Sarajevo Tourism Report 2024,pdf,partial


### 9.3 `isin` and `between`

In [40]:
# isin — match any value in a list
audio_or_video = df[df["source_type"].isin(["audio"])]
print(f"{len(audio_or_video)} audio documents")

# between — numeric range (endpoints inclusive by default)
midsize = df[df["file_size_kb"].between(100, 1000)]
print(f"{len(midsize)} documents between 100 and 1000 KB")

15 audio documents
13 documents between 100 and 1000 KB


### 9.4 String filtering with `.str`

In [41]:
# Titles that mention 'Sarajevo' (case-insensitive, tolerant of NaN)
has_sarajevo = df[df["title"].str.contains("sarajevo", case=False, na=False)]
has_sarajevo[["title", "source_type"]]

,title,source_type
0,Sarajevo Tourism Report 2024,pdf
17,Sarajevo Tourism Report 2024,pdf
19,Sarajevo Tourism Report 2024,pdf
46,Sarajevo Tourism Report 2024,pdf


In [42]:
# Filenames ending in .pdf
pdf_files = df[df["filename"].str.endswith(".pdf")]
print(f"{len(pdf_files)} .pdf filenames")

14 .pdf filenames


---
## 10 — Handling missing values

Real data always has holes. pandas represents them as `NaN` (numeric), `NaT` (datetimes), or `None` (objects). You **cannot** compare to `NaN` with `==`; always use `isna()` / `notna()`.

### 10.1 How many missing values per column?

In [43]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(1)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct":   missing_pct,
})
missing_report

,missing_count,missing_pct
_id,0,0.0
content.text,0,0.0
content.word_count,0,0.0
extracted_at,0,0.0
file_size_kb,2,3.3
filename,0,0.0
language,2,3.3
metadata.author,43,71.7
metadata.duration_sec,45,75.0
metadata.ocr_confidence,48,80.0


### 10.2 Why are those values missing?

Some of these missings are *structural* — they should be missing:

- `metadata.pages` is only defined for PDFs
- `metadata.duration_sec` is only defined for audio
- `metadata.ocr_confidence` is only defined for images with OCR text

Other missings are *real data-quality issues*: a few `file_size_kb` and `language` values that should have been recorded but weren't.

In [44]:
# Confirm the structural pattern: pages is non-null only for PDFs
pages_vs_type = df.groupby("source_type", observed=True)["metadata.pages"].agg(
    total="count",      # count() ignores NaN
    with_pages=lambda s: s.notna().sum(),
)
pages_vs_type

,total,with_pages
source_type,,
audio,0,0
image,0,0
pdf,14,14
web,0,0


### 10.3 Finding rows with any missing value

In [45]:
# Rows that have at least one NaN somewhere
rows_with_any_na = df[df.isna().any(axis=1)]
print(f"{len(rows_with_any_na)} rows have at least one missing value")
rows_with_any_na.head(3)

60 rows have at least one missing value


,_id,source_type,filename,title,extracted_at,file_size_kb,language,status,tags,content.text,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence,metadata.author
0,507f1f77bcf86cd799439000,pdf,sarajevo_tourism_report_2024.pdf,Sarajevo Tourism Report 2024,2025-09-04 10:13:00,1119.900024,de,processed,"[report, tourism]",Abstract: This paper explores NLP techniques f...,9,3.0,NaN,NaN,Alex Johnson
1,507f1f77bcf86cd799439001,image,product_label.png,product_label,2025-09-01 13:44:00,2100.500000,hr,partial,"[id, personal]",Best Before 12/2025 Made in BiH,6,NaN,NaN,0.906,None
2,507f1f77bcf86cd799439002,audio,interview_student.mp3,interview_student,2025-10-16 03:16:00,1686.300049,en,processed,[meeting],So my thesis is about multimodal deception det...,9,NaN,1311.6,NaN,None


We are **not cleaning** the data in this notebook — that is next week. Week 9 covers imputation, dropping, forward/backward fill, and deciding what to do with each missingness pattern.

---
## 11 — Regular expressions on text columns

Text columns — the bread and butter of unstructured data — often need pattern matching. pandas exposes this through the `.str` accessor, which takes regex patterns.

### 11.1 `str.contains` with regex

In [46]:
# Documents whose text contains a price in KM (number + KM)
price_pattern = r"\d+[.,]?\d*\s?KM"
has_price = df[df["content.text"].str.contains(price_pattern, regex=True, na=False)]
has_price[["source_type", "content.text"]]

,source_type,content.text
3,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025
4,image,Ćevapi 10KM Burek 8KM Kafa 2KM
14,image,STORE TOTAL 15.40 KM THANK YOU
17,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025
20,image,STORE TOTAL 15.40 KM THANK YOU
32,image,STORE TOTAL 15.40 KM THANK YOU
34,image,Ćevapi 10KM Burek 8KM Kafa 2KM
46,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025
59,image,Ćevapi 10KM Burek 8KM Kafa 2KM


### 11.2 `str.extract` — pull out the match

In [47]:
# Extract the first price value from each text
df["extracted_price"] = df["content.text"].str.extract(
    r"(\d+[.,]?\d*)\s?KM", expand=False
)

df.loc[df["extracted_price"].notna(), ["source_type", "content.text", "extracted_price"]]

,source_type,content.text,extracted_price
3,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025,245.60
4,image,Ćevapi 10KM Burek 8KM Kafa 2KM,10
14,image,STORE TOTAL 15.40 KM THANK YOU,15.40
17,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025,245.60
20,image,STORE TOTAL 15.40 KM THANK YOU,15.40
32,image,STORE TOTAL 15.40 KM THANK YOU,15.40
34,image,Ćevapi 10KM Burek 8KM Kafa 2KM,10
46,pdf,Ukupno za uplatu: 245.60 KM. Datum: 12.03.2025,245.60
59,image,Ćevapi 10KM Burek 8KM Kafa 2KM,10


### 11.3 `str.replace` — clean with regex

In [48]:
# Strip stray whitespace and collapse internal spaces in author names
df["metadata.author_clean"] = (
    df["metadata.author"]
      .str.strip()                         # leading/trailing spaces
      .str.replace(r"\s+", " ", regex=True)  # collapse internal whitespace
)

df[["metadata.author", "metadata.author_clean"]].dropna().head()

,metadata.author,metadata.author_clean
0,Alex Johnson,Alex Johnson
3,Alex Johnson,Alex Johnson
6,editor@klix.ba,editor@klix.ba
8,Amra S.,Amra S.
11,Staff Writer,Staff Writer


---
## 12 — Summary

What we did in this notebook:

1. **NumPy** — created arrays, inspected attributes, used vectorised operations, measured the speed-up over Python loops
2. **pandas `Series` and `DataFrame`** — built from lists, dicts, and (crucially) lists of dicts that mimic a MongoDB cursor
3. **Loading** — used `pd.json_normalize` to flatten MongoDB-style nested documents into a flat table
4. **Large files** — read a 50 000-row CSV in 10 000-row chunks and aggregated on the fly
5. **Memory** — downcast numerics, converted repeated strings to `category`
6. **Exploration** — `shape`, `dtypes`, `head`, `describe`, `info`, `value_counts`
7. **Selection** — `df["col"]`, `loc`, `iloc` and when each is appropriate
8. **Filtering** — boolean masks, `isin`, `between`, `.str.contains`, combinations with `& | ~`
9. **Missing values** — counted them, separated structural missings from data-quality missings
10. **Regex** — `.str.contains`, `.str.extract`, `.str.replace` on text columns

---

### What is next — Week 9: Data Cleaning

You have seen the mess — missing values, whitespace in author names, inconsistent strings, mixed languages. **Next week** we clean it: drop or impute missings, normalise strings, standardise categories, and produce a tidy analysis-ready DataFrame.

---
*IT 2012 – Unstructured Data · Week 8 · International Burch University*